# Travel Recommendation KNN V2

Notebook นี้ใช้สำหรับเทรนโมเดลแนะนำสถานที่ท่องเที่ยวแบบ Content-Based KNN ให้พร้อม export เป็นไฟล์ `.pkl` สำหรับนำไปใช้กับ FastAPI/Flutter

แนวทางของ V2:

- One-hot features: `region`, `province`, `category`, `type`, `activity`
- Text features: TF-IDF จาก `description`, `highlight`, `tags`, `nameTh`, `nameEn`
- ใช้ `NearestNeighbors(metric='cosine', algorithm='brute')` และเรียก `.fit()` จริง
- รองรับจังหวัดแบบ optional โดยเทรน 2 โมเดล: `model_with_province` และ `model_without_province`
- export artifact เป็น `.pkl` ที่มี model, encoders, vectorizer, metadata และ config ครบ
- มีฟังก์ชันตัวอย่างสำหรับ exclude สถานที่ที่เคยเห็นแล้ว เพื่อให้กดค้นหา input เดิมแล้วเจอสถานที่ใหม่ได้


## 1. ติดตั้ง/นำเข้า library

ถ้าเปิดใน Google Colab ปกติ library หลักจะมีอยู่แล้ว แต่ cell นี้ช่วยตรวจ version และ import ให้ครบ

In [ ]:
import json
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)


## 2. Upload dataset

ใน Colab ให้ upload ไฟล์ `dataset/attractions.json` จากโปรเจกต์ขึ้นมาก่อน

ไฟล์ในเครื่องโปรเจกต์อยู่ที่:

`C:\\flutter\\flutter-Project\\project\\dataset\\attractions.json`


In [ ]:
# ใช้ใน Google Colab เพื่อ upload attractions.json
try:
    from google.colab import files
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))
except Exception as error:
    print('Not running in Colab or upload skipped:', error)


## 3. โหลด dataset

ถ้าใช้ Colab แล้ว upload ไฟล์ไว้ที่ `/content/attractions.json` ให้ใช้ path นี้ได้เลย

In [ ]:
DATA_PATH = Path('/content/attractions.json')

# ถ้ารันในเครื่อง local จาก root project ให้ fallback path นี้
if not DATA_PATH.exists():
    DATA_PATH = Path('../dataset/attractions.json')

df = pd.read_json(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head(3))
print(df.columns.tolist())


## 4. เลือก column และทำความสะอาดข้อมูล

โมเดลใช้ข้อมูล 2 กลุ่ม:

1. Structured features: region, province, category, type, activity
2. Text features: nameTh, nameEn, description, highlight, tags


In [ ]:
required_columns = ['region', 'province', 'category', 'type', 'activity']
text_columns = ['nameTh', 'nameEn', 'description', 'highlight', 'tags']
metadata_columns = [
    'sourceRow', 'id', 'nameTh', 'nameEn', 'province', 'region', 'category',
    'type', 'activity', 'description', 'highlight', 'latitude', 'longitude'
]

for col in required_columns + text_columns:
    if col not in df.columns:
        df[col] = ''

model_df = df.copy()
for col in required_columns + ['nameTh', 'nameEn', 'description', 'highlight']:
    model_df[col] = model_df[col].fillna('').astype(str).str.strip()

if 'sourceRow' not in model_df.columns:
    model_df['sourceRow'] = np.arange(1, len(model_df) + 1)

metadata_df = model_df[[col for col in metadata_columns if col in model_df.columns]].copy()
print('Model rows:', len(model_df))
print('Missing values in core features:')
print(model_df[required_columns].replace('', np.nan).isna().sum())
display(model_df[['nameTh', 'province', 'region', 'category', 'type', 'activity']].head(10))


## 5. Helper สำหรับแยก activity/tags

`activity` ใน dataset เป็นหลายค่า เช่น `เรียนรู้ประวัติศาสตร์, ถ่ายรูป` จึงต้องแยกเป็น list ก่อนเข้า `MultiLabelBinarizer`


In [ ]:
def split_multi_value(value):
    if value is None:
        return []
    if isinstance(value, list):
        raw_items = value
    else:
        raw_items = re.split(r'[,/|;]+', str(value))
    items = []
    for item in raw_items:
        cleaned = str(item).strip().strip('"').strip("'")
        if cleaned:
            items.append(cleaned)
    return items

def normalize_tags(value):
    if value is None:
        return ''
    if isinstance(value, list):
        return ' '.join(str(item).strip() for item in value if str(item).strip())
    return str(value).strip()

activity_lists = model_df['activity'].apply(split_multi_value)
all_activities = sorted({item for row in activity_lists for item in row})
print('Activity count:', len(all_activities))
print(all_activities)

display(pd.DataFrame({
    'activity_original': model_df['activity'].head(10),
    'activity_split': activity_lists.head(10),
}))


## 6. สร้าง One-hot features

แยก encoder สำหรับ structured single-value และ multi-value activity เพื่อให้ตอน backend ใช้ซ้ำได้

In [ ]:
single_feature_columns = ['region', 'province', 'category', 'type']

onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
single_matrix = onehot_encoder.fit_transform(model_df[single_feature_columns])
single_feature_names = onehot_encoder.get_feature_names_out(single_feature_columns).tolist()

activity_encoder = MultiLabelBinarizer(sparse_output=True)
activity_matrix = activity_encoder.fit_transform(activity_lists)
activity_feature_names = [f'activity_{name}' for name in activity_encoder.classes_]

structured_matrix = hstack([single_matrix, activity_matrix], format='csr')
structured_feature_names = single_feature_names + activity_feature_names

print('Single matrix:', single_matrix.shape)
print('Activity matrix:', activity_matrix.shape)
print('Structured matrix:', structured_matrix.shape)
print('Structured feature sample:', structured_feature_names[:20])


## 7. สร้าง TF-IDF จาก description/highlight/tags

ภาษาไทยไม่มีการเว้นวรรคทุกคำเสมอไป จึงใช้ `analyzer='char_wb'` และ character n-gram เพื่อให้จับ keyword จากข้อความไทยได้โดยไม่ต้องติดตั้ง tokenizer เพิ่ม

In [ ]:
model_df['tags_text'] = model_df['tags'].apply(normalize_tags)
model_df['combined_text'] = (
    model_df['nameTh'].fillna('') + ' ' +
    model_df['nameEn'].fillna('') + ' ' +
    model_df['description'].fillna('') + ' ' +
    model_df['highlight'].fillna('') + ' ' +
    model_df['tags_text'].fillna('') + ' ' +
    model_df['category'].fillna('') + ' ' +
    model_df['type'].fillna('') + ' ' +
    model_df['activity'].fillna('')
).str.replace(r'\s+', ' ', regex=True).str.strip()

tfidf_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_features=1200,
    sublinear_tf=True,
)
text_matrix = tfidf_vectorizer.fit_transform(model_df['combined_text'])
text_feature_names = [f'tfidf_{term}' for term in tfidf_vectorizer.get_feature_names_out()]

print('Text matrix:', text_matrix.shape)
print('Text feature sample:', text_feature_names[:20])
display(model_df[['nameTh', 'combined_text']].head(3))


## 8. รวม feature และใส่ weight

Weight ช่วยบอกว่า feature ไหนสำคัญกว่ากัน โดยยังให้ text feature มีน้ำหนักเพื่อแยกอันดับละเอียดกว่าการ filter

In [ ]:
WEIGHTS = {
    'region': 2.0,
    'province': 2.5,
    'category': 3.0,
    'type': 3.0,
    'activity': 4.0,
    'text': 1.4,
}

def feature_weight(feature_name):
    if feature_name.startswith('region_'):
        return WEIGHTS['region']
    if feature_name.startswith('province_'):
        return WEIGHTS['province']
    if feature_name.startswith('category_'):
        return WEIGHTS['category']
    if feature_name.startswith('type_'):
        return WEIGHTS['type']
    if feature_name.startswith('activity_'):
        return WEIGHTS['activity']
    if feature_name.startswith('tfidf_'):
        return WEIGHTS['text']
    return 1.0

feature_matrix_raw = hstack([structured_matrix, text_matrix], format='csr')
feature_names = structured_feature_names + text_feature_names
weight_vector = np.array([feature_weight(name) for name in feature_names], dtype=np.float32)
feature_matrix = feature_matrix_raw.multiply(weight_vector).tocsr()

province_indices = np.array([i for i, name in enumerate(feature_names) if name.startswith('province_')])
no_province_indices = np.array([i for i, name in enumerate(feature_names) if not name.startswith('province_')])
feature_matrix_without_province = feature_matrix[:, no_province_indices].tocsr()

print('Full feature matrix:', feature_matrix.shape)
print('Without province matrix:', feature_matrix_without_province.shape)
print('Province feature count:', len(province_indices))


## 9. เทรน KNN จริงด้วย NearestNeighbors.fit()

จุดนี้คือส่วนสำคัญที่ทำให้ `.pkl` เป็นโมเดลที่พร้อมใช้งานจริง ไม่ใช่แค่เก็บ matrix เฉย ๆ

In [ ]:
max_neighbors = min(500, len(model_df))

model_with_province = NearestNeighbors(
    n_neighbors=max_neighbors,
    metric='cosine',
    algorithm='brute',
)
model_with_province.fit(feature_matrix)

model_without_province = NearestNeighbors(
    n_neighbors=max_neighbors,
    metric='cosine',
    algorithm='brute',
)
model_without_province.fit(feature_matrix_without_province)

print('KNN model with province fitted:', model_with_province)
print('KNN model without province fitted:', model_without_province)


## 10. ฟังก์ชันสร้าง user preference vector

ถ้า user ไม่เลือกจังหวัด จะใช้โมเดล `without_province` เพื่อไม่ให้จังหวัดกลายเป็น penalty

In [ ]:
def first_or_empty(values):
    if values is None:
        return ''
    if isinstance(values, str):
        return values.strip()
    values = [str(v).strip() for v in values if str(v).strip()]
    return values[0] if values else ''

def list_values(values):
    if values is None:
        return []
    if isinstance(values, str):
        return split_multi_value(values)
    return [str(v).strip() for v in values if str(v).strip()]

def build_user_vector(preference):
    region = first_or_empty(preference.get('region') or preference.get('regions'))
    province = first_or_empty(preference.get('province') or preference.get('provinces'))
    category = first_or_empty(preference.get('category') or preference.get('categories'))
    place_type = first_or_empty(preference.get('type') or preference.get('types'))
    activities = list_values(preference.get('activity') or preference.get('activities'))
    keywords = list_values(preference.get('keywords'))

    single_df = pd.DataFrame([{
        'region': region,
        'province': province,
        'category': category,
        'type': place_type,
    }])
    single_vec = onehot_encoder.transform(single_df[single_feature_columns])
    activity_vec = activity_encoder.transform([activities])
    structured_vec = hstack([single_vec, activity_vec], format='csr')

    query_text = ' '.join([region, province, category, place_type] + activities + keywords).strip()
    text_vec = tfidf_vectorizer.transform([query_text])
    full_vec = hstack([structured_vec, text_vec], format='csr').multiply(weight_vector).tocsr()

    has_province = bool(province)
    if has_province:
        return full_vec, True
    return full_vec[:, no_province_indices].tocsr(), False


## 11. ฟังก์ชัน recommend พร้อม exclude history

`exclude_source_rows` ใช้กรณี user เคยเห็น/เคยเปิดดูสถานที่แล้ว ถ้ากดค้นหา input เดิม ระบบสามารถข้ามรายการเดิมและแสดงสถานที่ใหม่ได้

In [ ]:
def recommend_places(preference, top_n=10, exclude_source_rows=None, candidate_pool=200):
    exclude_source_rows = set(exclude_source_rows or [])
    user_vec, has_province = build_user_vector(preference)
    model = model_with_province if has_province else model_without_province
    n_neighbors = min(max(candidate_pool, top_n), len(model_df))
    distances, indices = model.kneighbors(user_vec, n_neighbors=n_neighbors)

    rows = []
    for distance, idx in zip(distances[0], indices[0]):
        item = metadata_df.iloc[int(idx)].to_dict()
        source_row = int(item.get('sourceRow', int(idx) + 1))
        if source_row in exclude_source_rows:
            continue
        item['similarity'] = float(1 - distance)
        item['modelMode'] = 'with_province' if has_province else 'without_province'
        rows.append(item)
        if len(rows) >= top_n:
            break

    return pd.DataFrame(rows)


## 12. ทดสอบกรณีเลือกจังหวัดและไม่เลือกจังหวัด

In [ ]:
pref_without_province = {
    'region': 'ภาคใต้',
    'category': 'ธรรมชาติ',
    'type': 'จุดชมวิว',
    'activities': ['ชมวิว', 'ถ่ายรูป'],
    'keywords': ['ทะเล', 'ธรรมชาติ', 'วิวสวย'],
}

result_without_province = recommend_places(pref_without_province, top_n=10)
display(result_without_province[['sourceRow', 'nameTh', 'province', 'region', 'category', 'type', 'activity', 'similarity', 'modelMode']])


In [ ]:
pref_with_province = {
    'region': 'ภาคใต้',
    'province': 'ภูเก็ต',
    'category': 'ธรรมชาติ',
    'type': 'จุดชมวิว',
    'activities': ['ชมวิว', 'ถ่ายรูป'],
    'keywords': ['ทะเล', 'ธรรมชาติ', 'วิวสวย'],
}

result_with_province = recommend_places(pref_with_province, top_n=10)
display(result_with_province[['sourceRow', 'nameTh', 'province', 'region', 'category', 'type', 'activity', 'similarity', 'modelMode']])


## 13. ทดสอบกดค้นหา input เดิม แต่ exclude สถานที่ที่เคยเห็นแล้ว

ตัวอย่างนี้จำลองว่า user เห็น top 5 แรกแล้ว พอกดค้นหาอีกครั้ง backend ส่ง `exclude_source_rows` เข้าไป ผลลัพธ์จะเลื่อนไปสถานที่ใหม่

In [ ]:
first_page = recommend_places(pref_without_province, top_n=5)
seen_rows = first_page['sourceRow'].astype(int).tolist()

second_page = recommend_places(
    pref_without_province,
    top_n=10,
    exclude_source_rows=seen_rows,
)

print('Seen sourceRows:', seen_rows)
display(second_page[['sourceRow', 'nameTh', 'province', 'type', 'activity', 'similarity']])


## 14. ประเมินผลเบื้องต้นแบบ Preference Cases

Evaluation นี้เป็น rule-based relevance เพื่อใช้ประกอบรายงาน ไม่ใช่ accuracy แบบ classification

In [ ]:
evaluation_cases = [
    {'region': 'ภาคใต้', 'province': '', 'category': 'ธรรมชาติ', 'type': 'จุดชมวิว', 'activities': ['ชมวิว', 'ถ่ายรูป'], 'keywords': ['ทะเล', 'วิวสวย']},
    {'region': 'ภาคใต้', 'province': 'ภูเก็ต', 'category': 'ธรรมชาติ', 'type': 'ชายหาด', 'activities': ['พักผ่อน', 'ถ่ายรูป'], 'keywords': ['ทะเล', 'หาดทราย']},
    {'region': 'ภาคเหนือ', 'province': '', 'category': 'ประวัติศาสตร์และวัฒนธรรม', 'type': 'วัดและศาสนสถาน', 'activities': ['เรียนรู้วัฒนธรรม', 'ถ่ายรูป'], 'keywords': ['วัด', 'ศิลปะ']},
    {'region': 'ภาคเหนือ', 'province': 'เชียงใหม่', 'category': 'ธรรมชาติ', 'type': 'ภูเขา', 'activities': ['ชมวิว', 'เดินป่า'], 'keywords': ['ดอย', 'อากาศเย็น']},
]

def is_relevant(row, case):
    score = 0
    checks = 0
    for key in ['region', 'province', 'category', 'type']:
        expected = str(case.get(key, '')).strip()
        if not expected:
            continue
        checks += 1
        if str(row.get(key, '')).strip() == expected:
            score += 1
    expected_activities = set(case.get('activities') or [])
    if expected_activities:
        checks += 1
        row_activities = set(split_multi_value(row.get('activity', '')))
        if row_activities.intersection(expected_activities):
            score += 1
    return checks > 0 and score / checks >= 0.6

def evaluate_case(case, k=10):
    result = recommend_places(case, top_n=k)
    relevant_flags = result.apply(lambda row: is_relevant(row, case), axis=1)
    precision = relevant_flags.sum() / k
    # recall ใช้ relevant ใน dataset ตาม rule เดียวกันเป็น denominator
    relevant_all = metadata_df.apply(lambda row: is_relevant(row, case), axis=1).sum()
    recall = relevant_flags.sum() / relevant_all if relevant_all else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    hit_rate = 1 if relevant_flags.any() else 0
    return {
        'region': case.get('region', ''),
        'province': case.get('province', ''),
        'category': case.get('category', ''),
        'type': case.get('type', ''),
        'precision@10': precision,
        'recall@10': recall,
        'f1@10': f1,
        'hit_rate@10': hit_rate,
        'relevant_count': int(relevant_all),
    }

evaluation_df = pd.DataFrame([evaluate_case(case, k=10) for case in evaluation_cases])
display(evaluation_df)
print('Average evaluation:')
print(evaluation_df[['precision@10', 'recall@10', 'f1@10', 'hit_rate@10']].mean())


## 15. Export `.pkl` สำหรับ backend

ไฟล์นี้คือ artifact พร้อมใช้งานจริง เพราะมี fitted `NearestNeighbors` models แล้ว

In [ ]:
ARTIFACT_PATH = Path('travel_recommendation_knn_model_v2.pkl')

artifact = {
    'version': 'v2_onehot_tfidf_knn',
    'created_for': 'TravelRecommendation Flutter/Firebase project',
    'sklearn_version': sklearn.__version__,
    'model_with_province': model_with_province,
    'model_without_province': model_without_province,
    'feature_matrix': feature_matrix,
    'feature_matrix_without_province': feature_matrix_without_province,
    'metadata': metadata_df,
    'onehot_encoder': onehot_encoder,
    'activity_encoder': activity_encoder,
    'tfidf_vectorizer': tfidf_vectorizer,
    'feature_names': feature_names,
    'no_province_indices': no_province_indices,
    'weights': WEIGHTS,
    'single_feature_columns': single_feature_columns,
    'text_columns': text_columns,
    'notes': (
        'Use model_with_province when user selected province. '
        'Use model_without_province when province is optional/empty. '
        'Use exclude_source_rows/history in backend to show new places for repeated same input.'
    ),
}

with open(ARTIFACT_PATH, 'wb') as f:
    pickle.dump(artifact, f)

print('Exported:', ARTIFACT_PATH.resolve())
print('File size MB:', ARTIFACT_PATH.stat().st_size / 1024 / 1024)


## 16. Download `.pkl` จาก Colab

หลังรัน cell export แล้ว ให้รัน cell นี้เพื่อดาวน์โหลดไฟล์ลงเครื่อง แล้วนำไฟล์ไปวางที่:

`C:\\flutter\\flutter-Project\\project\\backend\\models\\travel_recommendation_knn_model_v2.pkl`


In [ ]:
try:
    from google.colab import files
    files.download(str(ARTIFACT_PATH))
except Exception as error:
    print('Download is available only in Colab:', error)


## 17. สรุปสำหรับรายงาน

ข้อความที่ใช้เขียนรายงานได้:

โมเดลแนะนำสถานที่ท่องเที่ยวใช้แนวทาง Content-Based KNN โดยนำข้อมูลความสนใจของผู้ใช้ ได้แก่ ภูมิภาค จังหวัด หมวดหมู่ ประเภทสถานที่ และกิจกรรม มาแปลงเป็นคุณลักษณะเชิงตัวเลขด้วย One-Hot Encoding และ Multi-Label Encoding สำหรับกิจกรรมหลายค่า จากนั้นนำข้อมูลคำอธิบาย จุดเด่น และ tag ของสถานที่มาสร้างคุณลักษณะข้อความด้วย TF-IDF แบบ character n-gram เพื่อให้ระบบสามารถพิจารณาความหมายของข้อมูลสถานที่ได้ละเอียดกว่าการกรองข้อมูลตามหมวดหมู่เพียงอย่างเดียว โมเดลถูกเทรนด้วย NearestNeighbors โดยใช้ cosine distance เป็นตัววัดความคล้ายคลึง และ export เป็นไฟล์ `.pkl` สำหรับนำไปใช้งานผ่าน backend API
